In [3]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install "transformers[torch]"

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration


In [2]:

train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")


In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:

train_data.sample(10)

,id,dialogue,summary
10033,13864690,Matthew: Booked the tickets!\nMary: Thanks!\nJ...,"Matthew booked the tickets for Mary, Jay and h..."
1953,13730387,Luis: kisses\r\nAnna: yo! how is my bro? tell ...,"Anna learns that Theo, Luis' friend from Wawa,..."
12008,13810004,"Donna: Hi Rach, everyone ok your end?\r\nRache...",Rachel and Donna talk about their sons' grades...
5217,13864531,Becky: What's the wifi password?\nSylvia: <fil...,Becky is on the 4th floor. There are fewer WIF...
12772,13680122,"Amy: So, at what time tomorrow?\r\nJack: Hmmmm...",Amy and Jack will meet at 5:30 PM at Jack's pl...
12493,13864560,Alice: Have you read anything by Alice Munro?\...,Cecil read short stories by Alice Munro a few ...
8340,13829551,Xavier: what scenario we will be doing next we...,Xavier and Peter have at least 3 scenarios to ...
6731,13680653,Carmen: Later on can I ask u for more help? bu...,Gemma will help Carmen later.
12891,13727784,Felix: Could you kindly confirm your arrival?\...,Gabriel confirmed his arrival on Felix's request.
4470,13682153,Stan: May I stay home?\r\nCarol: No. \r\nStan:...,Stan wants to stay home because he has a test ...


In [5]:
train_data.shape

(14732, 3)

In [6]:
val_data.shape

(818, 3)

In [10]:
# random sampling --

train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)


# Preprocessing --

In [12]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.strip().lower()
    return text



In [13]:

train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summmary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = train_data["dialogue"].apply(clean_data)
val_data["summmary"] = train_data["summary"].apply(clean_data)


In [15]:
train_data["dialogue"][0]

"darren: hi buddy! didn't see you at the gym?! you ok? ryan: yeah, mate. been working bloody nights, too knackered to do much! darren: nightmare! you coming friday? ryan: nah, working again, aren't it!? management don't give a shit about us lot in the warehouse! darren: yeah, but it's double time, ain't it! you'll be coining it in! see you next week then! ryan: maybe, i dunno... if i can be arsed! i'll give you a bell. see you, mate."

# Tokenization--


In [16]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [19]:
# raw data => tokenized inp for => fine tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)

    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"]
    return inputs
    

In [20]:

train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()


In [22]:
train_dataset[0]

{'input_ids': [649, 1536, 10, 7102, 23707, 55, 737, 31, 17, 217, 25, 44, 8, 7868, 10769, 25, 3, 1825, 58, 3, 651, 152, 10, 17945, 6, 3, 5058, 5, 118, 464, 1717, 63, 8348, 6, 396, 30302, 3737, 12, 103, 231, 55, 649, 1536, 10, 19400, 55, 25, 1107, 9030, 1135, 58, 3, 651, 152, 10, 3, 8607, 6, 464, 541, 6, 33, 29, 31, 17, 34, 55, 58, 758, 278, 31, 17, 428, 3, 9, 3, 7, 10536, 81, 178, 418, 16, 8, 11625, 55, 649, 1536, 10, 17945, 6, 68, 34, 31, 7, 1486, 97, 6, 3, 9, 77, 31, 17, 34, 55, 25, 31, 195, 36, 7485, 53, 34, 16, 55, 217, 25, 416, 471, 258, 55, 3, 651, 152, 10, 2087, 6, 3, 23, 146, 29, 29, 32, 233, 3, 99, 3, 23, 54, 36, 1584, 3843, 55, 3, 23, 31, 195, 428, 25, 3, 9, 12815, 5, 217, 25, 6, 3, 5058, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [25]:
# input ids - dialogue => token ids

#1 => EOS, 0=> padding

# attention mask
# labels - target => summary token

len(train_dataset[0]["input_ids"])


512

In [26]:
type(train_dataset)
type(val_dataset)

list

# Working With Our Model -


In [27]:
# training is done !

# NLP => generation task-

model = T5ForConditionalGeneration.from_pretrained("t5-small")


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [29]:
# fine - tune =>

import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else :
    device = torch.device("cpu")

print("device: ",device)
model.to(device)


device:  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [31]:

# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=4,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500
)

In [32]:

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)